# Sales Forecasting Project

## Objective
This notebook implements a predictive model for retail sales forecasting using gradient boosting techniques. The goal is to predict future demand based on historical patterns, promotional activities, and pricing strategies.


## Environment Setup
Importing necessary libraries for data manipulation, visualization, and machine learning.


In [357]:
!pip install catboost

In [358]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from catboost import CatBoostRegressor
from copy import deepcopy
import seaborn as sns
import matplotlib.pyplot as plt

print("Libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")



Libraries loaded successfully
Pandas version: 2.2.2
NumPy version: 2.0.2


## Data Acquisition
Loading the training dataset from CSV file. The dataset contains historical sales records with associated features.


In [359]:
from google.colab import drive, files
drive.mount('/content/drive')
def load_dataset(filepath):
    df = pd.read_csv(filepath, delimiter=',')
    df['period_start_dt'] = pd.to_datetime(df['period_start_dt'], format="%Y-%m-%d")
    df.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
    print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
    return df

dataset = load_dataset('/content/drive/MyDrive/train.csv')
dataset.head()



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded: 35344 rows, 11 columns


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



,id,product_rk,store_location_rk,period_start_dt,demand,PROMO1_FLAG,PROMO2_FLAG,PRICE_REGULAR,PRICE_AFTER_DISC,NUM_CONSULTANT,AUTORIZATION_FLAG
0,0,40369,309,2016-12-19,29.0,NaN,NaN,NaN,NaN,NaN,NaN
1,1,40370,309,2016-12-19,64.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2,40372,309,2016-12-19,32.0,NaN,NaN,NaN,NaN,NaN,NaN
3,3,40373,309,2016-12-19,10.0,NaN,NaN,NaN,NaN,NaN,NaN
4,4,46272,309,2016-12-19,15.0,NaN,NaN,NaN,NaN,NaN,NaN


## Initial Data Exploration

### Dataset Overview
Examining the structure, data types, and basic statistics of our dataset.


In [360]:
print("="*50)
print("DATASET INFORMATION")
print("="*50)
print(f"\nShape: {dataset.shape}")
print(f"\nColumns: {list(dataset.columns)}")
print(f"\nData types:\n{dataset.dtypes}")
print(f"\nMemory usage: {dataset.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
dataset.describe()



DATASET INFORMATION

Shape: (35344, 11)

Columns: ['id', 'product_rk', 'store_location_rk', 'period_start_dt', 'demand', 'PROMO1_FLAG', 'PROMO2_FLAG', 'PRICE_REGULAR', 'PRICE_AFTER_DISC', 'NUM_CONSULTANT', 'AUTORIZATION_FLAG']

Data types:
id                            int64
product_rk                    int64
store_location_rk             int64
period_start_dt      datetime64[ns]
demand                      float64
PROMO1_FLAG                 float64
PROMO2_FLAG                 float64
PRICE_REGULAR               float64
PRICE_AFTER_DISC            float64
NUM_CONSULTANT              float64
AUTORIZATION_FLAG           float64
dtype: object

Memory usage: 2.97 MB


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



,id,product_rk,store_location_rk,period_start_dt,demand,PROMO1_FLAG,PROMO2_FLAG,PRICE_REGULAR,PRICE_AFTER_DISC,NUM_CONSULTANT,AUTORIZATION_FLAG
count,35344.000000,35344.000000,35344.000000,35344,34144.000000,35159.000000,35159.0,34217.000000,34212.000000,35159.0,35159.000000
mean,17766.554012,49253.732232,844.240154,2018-07-24 16:17:19.746491904,12.245636,0.206434,0.0,1167.679357,1155.778351,0.0,0.907677
min,0.000000,40369.000000,309.000000,2016-12-19 00:00:00,0.000000,0.000000,0.0,49.000000,8.647059,0.0,0.000000
25%,8881.750000,40370.000000,535.000000,2017-11-06 00:00:00,2.000000,0.000000,0.0,284.290000,199.000000,0.0,1.000000
50%,17770.500000,40372.000000,862.000000,2018-07-30 00:00:00,6.000000,0.000000,0.0,1000.000000,1000.000000,0.0,1.000000
75%,26647.250000,46272.000000,1173.000000,2019-04-22 00:00:00,12.000000,0.000000,0.0,2000.000000,2000.000000,0.0,1.000000
max,35541.000000,96212.000000,1380.000000,2019-12-30 00:00:00,1160.000000,2.000000,0.0,3000.000000,3000.000000,0.0,1.000000
std,10258.040738,19145.064867,333.229160,NaN,32.604642,0.433393,0.0,1046.828551,1057.912830,0.0,0.289486


### Missing Value Analysis
Identifying and quantifying missing data across all features.


In [361]:
missing_analysis = dataset.isnull().sum()
missing_pct = (missing_analysis / len(dataset)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_analysis,
    'Percentage': missing_pct
}).sort_values('Missing_Count', ascending=False)

print("\nMissing Value Summary:")
print(missing_df[missing_df['Missing_Count'] > 0])




Missing Value Summary:
                   Missing_Count  Percentage
demand                      1200    3.395201
PRICE_AFTER_DISC            1132    3.202807
PRICE_REGULAR               1127    3.188660
NUM_CONSULTANT               185    0.523427
PROMO2_FLAG                  185    0.523427
AUTORIZATION_FLAG            185    0.523427
PROMO1_FLAG                  185    0.523427


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



## Data Visualization Functions

### Time Series Visualization Utility
Creating reusable visualization functions for exploratory data analysis.


In [362]:
def visualize_history(ts_df, groupby_columns, time_column, target_column, ts_num = 10, aggregation_method = 'sum'):
  if groupby_columns is None:
    ts_df[target_column + time_column + 'const'] = 1
    groupby_columns = [target_column + time_column + 'const']

  pivot_ts = ts_df.groupby(groupby_columns + [time_column]).agg(aggregation_method)

  index_column_name = ', '.join([groupby_columns[i]+'={0['+str(i)+']}' for i in range(len(groupby_columns))])
  pivot_ts.index = [pivot_ts.index.map(index_column_name.format) , pivot_ts.index.get_level_values(len(groupby_columns))]

  pivot_ts = pivot_ts.unstack([0])[target_column]

  fig = go.Figure()
  for col in pivot_ts.columns[:ts_num]:
      fig.add_trace(go.Scatter(x=pivot_ts.index, y=pivot_ts[col], mode='lines', name=str(col)))

  fig.update_layout(
      height=350,
      width=1300,
      title="First {0} time series for {1} variable".format(ts_num, target_column),
      xaxis_title=time_column,
      yaxis_title=target_column + ' value',
      legend_title='Time series ID columns: ' + ', '.join(groupby_columns)
  )


  return fig

visualize_history(dataset, ['product_rk'], 'period_start_dt', 'demand', ts_num = 3)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



### Demand Visualization by Product
Visualizing historical demand patterns across different products.


In [363]:
print("Generating demand visualizations...")
visualize_history(dataset,
                groupby_columns=['product_rk'],
                time_column='period_start_dt',
                target_column='demand',
                ts_num=6,
                aggregation_method='sum')



Generating demand visualizations...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



## Feature Engineering Pipeline

### Data Quality Assessment
Evaluating feature completeness and identifying variables for preprocessing.


In [364]:
feature_analysis = {}
for col in dataset.columns:
    if dataset[col].dtype in ['float64', 'int64']:
        feature_analysis[col] = {
            'unique_values': dataset[col].nunique(),
            'missing_pct': (dataset[col].isnull().sum() / len(dataset)) * 100,
            'zero_count': (dataset[col] == 0).sum()
        }

print("\nNumerical Feature Analysis:")
for feat, stats in feature_analysis.items():
    print(f"\n{feat}:")
    print(f"  Unique values: {stats['unique_values']}")
    print(f"  Missing: {stats['missing_pct']:.2f}%")
    print(f"  Zeros: {stats['zero_count']}")




Numerical Feature Analysis:

id:
  Unique values: 35344
  Missing: 0.00%
  Zeros: 1

product_rk:
  Unique values: 6
  Missing: 0.00%
  Zeros: 0

store_location_rk:
  Unique values: 41
  Missing: 0.00%
  Zeros: 0

demand:
  Unique values: 1806
  Missing: 3.40%
  Zeros: 5454

PROMO1_FLAG:
  Unique values: 3
  Missing: 0.52%
  Zeros: 28323

PROMO2_FLAG:
  Unique values: 1
  Missing: 0.52%
  Zeros: 35159

PRICE_REGULAR:
  Unique values: 229
  Missing: 3.19%
  Zeros: 0

PRICE_AFTER_DISC:
  Unique values: 1036
  Missing: 3.20%
  Zeros: 0

NUM_CONSULTANT:
  Unique values: 1
  Missing: 0.52%
  Zeros: 35159

AUTORIZATION_FLAG:
  Unique values: 2
  Missing: 0.52%
  Zeros: 3246


### Feature Cleaning
Removing features with insufficient variability that would not contribute to model performance.


In [365]:
print("Initial columns:", dataset.columns.tolist())
print(f"\nUnique values in PROMO1_FLAG: {dataset['PROMO1_FLAG'].unique()}")
print(f"Unique values in PROMO2_FLAG: {dataset['PROMO2_FLAG'].unique()}")
print(f"Unique values in NUM_CONSULTANT: {dataset['NUM_CONSULTANT'].unique()}")

dataset = dataset.drop(['PROMO2_FLAG', 'NUM_CONSULTANT'], axis=1)
print(f"\nColumns after cleaning: {dataset.columns.tolist()}")
print(f"Dataset shape: {dataset.shape}")



Initial columns: ['id', 'product_rk', 'store_location_rk', 'period_start_dt', 'demand', 'PROMO1_FLAG', 'PROMO2_FLAG', 'PRICE_REGULAR', 'PRICE_AFTER_DISC', 'NUM_CONSULTANT', 'AUTORIZATION_FLAG']

Unique values in PROMO1_FLAG: [nan  1.  0.  2.]
Unique values in PROMO2_FLAG: [nan  0.]
Unique values in NUM_CONSULTANT: [nan  0.]

Columns after cleaning: ['id', 'product_rk', 'store_location_rk', 'period_start_dt', 'demand', 'PROMO1_FLAG', 'PRICE_REGULAR', 'PRICE_AFTER_DISC', 'AUTORIZATION_FLAG']
Dataset shape: (35344, 9)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



## Missing Value Imputation Strategy

### Methodology
Implementing aggregation-based imputation using mean values across stores for each product-date combination.


In [366]:
def fill_missing_agg(ts_df,column_name, ts_id):
  values = dataset.set_index(ts_id)\
            .unstack([0,1])\
            [column_name].\
              mean()
  new_ts_df = ts_df.set_index(ts_id).\
    merge(ts_df.set_index(ts_id)\
              .unstack([0,1])\
              [column_name].\
              fillna(value = values).\
              stack([1,0], future_stack=True).\
              rename(column_name),
          how = 'left', right_index = True, left_index = True)\
    .reset_index()
  del new_ts_df[column_name+'_x']
  return new_ts_df.rename(columns = {column_name+'_y':column_name})
dataset = fill_missing_agg(ts_df = dataset,column_name = 'PRICE_REGULAR' , ts_id= ['product_rk', 'period_start_dt', 'store_location_rk'])


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



### Applying Imputation
Filling missing values in price and authorization features.


In [367]:
print("Applying imputation to PRICE_REGULAR...")
dataset = fill_missing_agg(ts_df=dataset,
                          column_name='PRICE_REGULAR',
                          ts_id=['product_rk', 'period_start_dt', 'store_location_rk'])
print(f"Missing values in PRICE_REGULAR: {dataset['PRICE_REGULAR'].isnull().sum()}")

print("\nApplying imputation to PRICE_AFTER_DISC...")
dataset = fill_missing_agg(ts_df=dataset,
                          column_name='PRICE_AFTER_DISC',
                          ts_id=['product_rk', 'period_start_dt', 'store_location_rk'])
print(f"Missing values in PRICE_AFTER_DISC: {dataset['PRICE_AFTER_DISC'].isnull().sum()}")

print("\nApplying imputation to AUTORIZATION_FLAG...")
dataset = fill_missing_agg(ts_df=dataset,
                          column_name='AUTORIZATION_FLAG',
                          ts_id=['product_rk', 'period_start_dt', 'store_location_rk'])
print(f"Missing values in AUTORIZATION_FLAG: {dataset['AUTORIZATION_FLAG'].isnull().sum()}")

print("\nImputation complete!")
print(f"Total missing values: {dataset.isnull().sum().sum()}")



Applying imputation to PRICE_REGULAR...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



Missing values in PRICE_REGULAR: 175

Applying imputation to PRICE_AFTER_DISC...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



Missing values in PRICE_AFTER_DISC: 175

Applying imputation to AUTORIZATION_FLAG...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



Missing values in AUTORIZATION_FLAG: 175

Imputation complete!
Total missing values: 1910


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



### Data Filtering
Removing records from problematic store location that exhibits anomalous patterns.


In [368]:
print(f"Rows before filtering: {len(dataset)}")
print(f"Unique stores: {dataset['store_location_rk'].nunique()}")

dataset = dataset[dataset['store_location_rk'] != 309]

print(f"\nRows after filtering: {len(dataset)}")
print(f"Unique stores remaining: {dataset['store_location_rk'].nunique()}")
print(f"\nFinal missing value check:\n{dataset.isnull().sum()}")



Rows before filtering: 35344
Unique stores: 41

Rows after filtering: 35329
Unique stores remaining: 40

Final missing value check:
product_rk              0
period_start_dt         0
store_location_rk       0
id                      0
demand               1200
PROMO1_FLAG           170
PRICE_REGULAR         170
PRICE_AFTER_DISC      170
AUTORIZATION_FLAG     170
dtype: int64


## Temporal Feature Engineering

### Calendar Features
Extracting temporal components from date to capture seasonal patterns.


In [369]:
dataset["ind_of_year"] = dataset.period_start_dt.dt.year
dataset["ind_of_month"] = dataset.period_start_dt.dt.month
dataset["ind_of_day"] = dataset.period_start_dt.dt.day

print("Temporal features created:")
print(f"Years: {sorted(dataset['ind_of_year'].unique())}")
print(f"Months: {sorted(dataset['ind_of_month'].unique())}")
print(f"Day range: {dataset['ind_of_day'].min()} to {dataset['ind_of_day'].max()}")

dataset.head()



Temporal features created:
Years: [np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019)]
Months: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
Day range: 1 to 31


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



,product_rk,period_start_dt,store_location_rk,id,demand,PROMO1_FLAG,PRICE_REGULAR,PRICE_AFTER_DISC,AUTORIZATION_FLAG,ind_of_year,ind_of_month,ind_of_day
15,40369,2016-12-19,317,15,50.0,NaN,NaN,NaN,NaN,2016,12,19
16,40370,2016-12-19,317,16,44.0,NaN,NaN,NaN,NaN,2016,12,19
17,40372,2016-12-19,317,17,13.0,NaN,NaN,NaN,NaN,2016,12,19
18,40373,2016-12-19,317,18,6.0,NaN,NaN,NaN,NaN,2016,12,19
19,46272,2016-12-19,317,19,34.0,NaN,NaN,NaN,NaN,2016,12,19


## Advanced Feature Engineering

### Lag Feature Generation
Creating lagged features with rolling windows and exponential moving averages to capture historical trends.


In [370]:
from ipywidgets import IntProgress
from itertools import product
def percentile(n):
    def percentile_(x):
        return np.percentile(x, n)
    percentile_.__name__ = 'pctl%s' % n
    return percentile_
def fill_missing_dates(x, date_col, freq = None, default_value = np.nan):
    if freq is None:
        try:
           freq = pd.infer_freq(x.set_index(date_col).index[:min(100, x.shape[0])])
        except:
           freq = 'D'
        if freq is None:
          freq = 'D'
          Warning('TS freq is not defined! Daily granularity is provided!')
    idx = pd.date_range(x[date_col].min(), x[date_col].max(), freq=freq)
    results = x.set_index(date_col).reindex(idx,fill_value = default_value)
    results.index.rename(date_col, inplace = True)
    return results.reset_index()
def calc_preag_fill(data, group_col, date_col, target_cols, preagg_method):
    data_preag = data.groupby(group_col).agg(
        preagg_method)[target_cols].reset_index()
    data_preag_filled = data_preag.groupby(group_col[:-1]).apply(
         fill_missing_dates, date_col=date_col).drop(group_col[:-1],
                                                     axis=1).reset_index()
    return data_preag
def calc_rolling(data_preag_filled, group_col, date_col, method, w):
    lf_df_filled = data_preag_filled.groupby(group_col[:-1]).\
        apply(lambda x: x.set_index(date_col).rolling(window=w, min_periods=1).agg(method)).drop(group_col[:-1], axis=1).reset_index(group_col)
    return lf_df_filled
def calc_ewm(data_preag_filled, group_col, date_col, span):
    lf_df_filled = data_preag_filled.groupby(group_col[:-1]).\
        apply(lambda x: x.set_index(date_col).ewm(span=span).mean()).drop(group_col[:-1], axis=1).reset_index(group_col)
    return lf_df_filled
def shift(lf_df_filled, group_col, date_col, lag, kwargs = None):
    lf_df = (lf_df_filled.
        set_index(date_col).
        groupby(group_col[:-1]).
        apply(lambda x: x.shift(lag, kwargs)).
        drop(group_col[:-1], axis=1).
        reset_index()
    )
    return lf_df
def add_lag_features(
        data: pd.DataFrame,
        target_cols: list = ['Demand'],
        id_cols: list = ['SKU_id', 'Store_id'],
        date_col: str = 'Date',
        lags: list = [7, 14, 21, 28],
        windows: list = ['7D', '14D', '28D', '56D'],
        preagg_methods: list = ['mean'],
        agg_methods: list = ['mean', 'median', percentile(10), pd.Series.skew],
        dynamic_filters: list = ['weekday', 'Promo'],
        ewm_params: dict = {'weekday': [14, 28], 'Promo': [14, 42]}) -> pd.DataFrame:

    data = data.sort_values(date_col)
    out_df = deepcopy(data)
    dates = [min(data[date_col]), max(data[date_col])]
    total = len(target_cols) * len(lags) * len(windows) * len(preagg_methods) * len(agg_methods) * len(dynamic_filters)
    progress = IntProgress(min=0, max=total)
    display(progress)
    for filter_col in dynamic_filters:
        group_col = [filter_col] + id_cols + [date_col]
        for preagg in preagg_methods:
          data_preag_filled = calc_preag_fill(data, group_col, date_col,
                                                  target_cols, preagg)
          for alpha in ewm_params.get(filter_col, []):
              ewm_filled = calc_ewm(data_preag_filled, group_col,
                                    date_col, alpha)
              for lag in lags:
                ewm = shift(ewm_filled, group_col, date_col, lag)
                new_names = {x: "{0}_lag{1}d_alpha{2}_key{3}_preag{4}_{5}_dynamic_ewm".\
                    format(x, lag, alpha, '&'.join(id_cols), preagg, filter_col) for x in target_cols}
                out_df = pd.merge(out_df,
                                  ewm.rename(columns=new_names),
                                  how='left',
                                  on=group_col)
          for w in windows:
              for method in agg_methods:
                  rolling_filled = calc_rolling(data_preag_filled,
                                                group_col, date_col,
                                                method, w)
                  for lag in lags:
                    rolling = shift(rolling_filled, group_col, date_col, lag)
                    method_name = method.__name__ if type(
                        method) != str else method
                    new_names = {x: "{0}_lag{1}d_w{2}_key{3}_preag{4}_ag{5}_{6}_dynamic_rolling".\
                                  format(x, lag, w, '&'.join(id_cols), preagg, method_name, filter_col) for x in target_cols}
                    out_df = pd.merge(out_df,
                                      rolling.rename(columns=new_names),
                                      how='left',
                                      on=group_col)
                    progress.value += 1
    return out_df


In [371]:
print("Configuring lag feature parameters...")
print("="*50)

dataset['NoFilter'] = 1
target_cols = ['demand']
id_cols = ['product_rk', 'store_location_rk']
date_col = 'period_start_dt'
lags = [5, 6, 7, 8, 12, 16]
windows = ['4D','8D','12D']

preagg_methods = ['mean']
agg_methods = ['mean', 'median']
dynamic_filters = ['NoFilter', 'PROMO1_FLAG'] if 'PROMO1_FLAG' in dataset.columns else ['NoFilter']

ewm_params = {'NoFilter':[4,8], 'PROMO1_FLAG':[8]}

print(f"Target columns: {target_cols}")
print(f"ID columns: {id_cols}")
print(f"Lag values: {lags}")
print(f"Window sizes: {windows}")
print(f"Dynamic filters: {dynamic_filters}")
print("\nGenerating lag features...")

Configuring lag feature parameters...
Target columns: ['demand']
ID columns: ['product_rk', 'store_location_rk']
Lag values: [5, 6, 7, 8, 12, 16]
Window sizes: ['4D', '8D', '12D']
Dynamic filters: ['NoFilter', 'PROMO1_FLAG']

Generating lag features...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



In [372]:
import warnings
warnings.filterwarnings('ignore')

dataset = add_lag_features(
    data=dataset,
    target_cols=target_cols,
    id_cols=id_cols,
    date_col=date_col,
    lags=lags,
    windows=windows,
    preagg_methods=preagg_methods,
    agg_methods=agg_methods,
    dynamic_filters=dynamic_filters,
    ewm_params=ewm_params
)

warnings.filterwarnings('default')

print(f"\nFeature engineering complete!")
print(f"Total features: {len(dataset.columns)}")
print(f"Dataset shape: {dataset.shape}")

IntProgress(value=0, max=72)


Feature engineering complete!
Total features: 103
Dataset shape: (35329, 103)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



## Train-Validation Split

### Temporal Split Strategy
Splitting data chronologically to maintain temporal integrity and prevent data leakage.


In [373]:
train_subset = dataset[~dataset['demand'].isnull()].copy()
inference_data  = dataset[dataset['demand'].isnull()].copy()

print(f"Training data: {len(train_subset)} records")
print(f"Inference data: {len(inference_data)} records")

split_date = pd.to_datetime('2019-11-04')
print(f"\nSplit date: {split_date}")

training_split = train_subset[train_subset['period_start_dt'] < split_date].copy()
validation_split = train_subset[train_subset['period_start_dt'] >= split_date].copy()

print(f"\nTraining set: {len(training_split)} records")
print(f"Validation set: {len(validation_split)} records")
print(f"Split ratio: {len(validation_split)/len(train_subset):.2%} validation")


Training data: 34129 records
Inference data: 1200 records

Split date: 2019-11-04 00:00:00

Training set: 33169 records
Validation set: 960 records
Split ratio: 2.81% validation


In [374]:
features_train = training_split.drop(['id','demand','period_start_dt'], axis=1)
target_train = training_split['demand']

features_valid = validation_split.drop(['id','demand','period_start_dt'], axis=1)
target_valid = validation_split['demand']

print(f"Training features shape: {features_train.shape}")
print(f"Training target shape: {target_train.shape}")
print(f"Validation features shape: {features_valid.shape}")
print(f"Validation target shape: {target_valid.shape}")
print(f"\nFeature count: {features_train.shape[1]}")


Training features shape: (33169, 100)
Training target shape: (33169,)
Validation features shape: (960, 100)
Validation target shape: (960,)

Feature count: 100


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



## Model Training

### CatBoost Gradient Boosting
Training a CatBoost regressor optimized for MAE with early stopping to prevent overfitting.


In [375]:
print("Initializing CatBoost model...")
print("="*50)

cb_model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    loss_function='MAE',
    eval_metric='MAE',
    verbose=200,
    random_seed=1,
)

print("\nModel parameters:")
print(f"  Iterations: 2000")
print(f"  Learning rate: 0.05")
print(f"  Depth: 8")
print(f"  Loss function: MAE")
print(f"\nTraining model...")

cb_model.fit(
    features_train, target_train,
    eval_set=(features_valid, target_valid),
    early_stopping_rounds=100,
    use_best_model=True
)

print("\nTraining complete!")

Initializing CatBoost model...

Model parameters:
  Iterations: 2000
  Learning rate: 0.05
  Depth: 8
  Loss function: MAE

Training model...
0:	learn: 10.1115367	test: 5.4520287	best: 5.4520287 (0)	total: 131ms	remaining: 4m 22s
200:	learn: 6.8765657	test: 3.5767821	best: 3.5672003 (197)	total: 20.6s	remaining: 3m 4s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.567200326
bestIteration = 197

Shrink model to first 198 iterations.

Training complete!


### Model Evaluation
Assessing model performance on validation set.


In [376]:
predictions = cb_model.predict(features_valid)
mae_score = mean_absolute_error(target_valid, predictions)

print(f"Validation MAE: {mae_score:.4f}")
print(f"\nPrediction statistics:")
print(f"  Mean: {predictions.mean():.2f}")
print(f"  Std: {predictions.std():.2f}")
print(f"  Min: {predictions.min():.2f}")
print(f"  Max: {predictions.max():.2f}")

print(f"\nActual statistics:")
print(f"  Mean: {target_valid.mean():.2f}")
print(f"  Std: {target_valid.std():.2f}")

mae_score

Validation MAE: 3.5672

Prediction statistics:
  Mean: 5.35
  Std: 5.06
  Min: -0.09
  Max: 27.16

Actual statistics:
  Mean: 7.00
  Std: 9.77


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



3.5672013257058524

## Inference and Submission

### Generating Predictions
Applying trained model to test set and preparing submission file.


In [377]:
X_test_full = inference_data.drop(['id','period_start_dt'], axis=1)

print(f"Inference data shape: {X_test_full.shape}")

final_predictions = cb_model.predict(X_test_full)

print(f"\nPredictions generated: {len(final_predictions)}")
print(f"Prediction range: [{final_predictions.min():.2f}, {final_predictions.max():.2f}]")

Inference data shape: (1200, 101)

Predictions generated: 1200
Prediction range: [-0.47, 67.10]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



In [378]:
final_predictions = np.where(final_predictions < 0, 0, final_predictions)
final_predictions = np.round(final_predictions, 3)

print(f"Negative predictions corrected: {(final_predictions == 0).sum()}")

y_results = inference_data[['id']].copy()
y_results['predicted'] = final_predictions

print(f"\nSubmission file prepared:")
print(f"  Rows: {len(y_results)}")
print(f"  Columns: {y_results.columns.tolist()}")

y_results.head()

Negative predictions corrected: 9

Submission file prepared:
  Rows: 1200
  Columns: ['id', 'predicted']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



,id,predicted
34129,25034,0.877
34130,25035,2.595
34131,23184,10.101
34132,21336,0.000
34133,7409,11.908


In [379]:
y_results.to_csv('submission.csv', index=False)
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>